# Assign topic label using LLM

The aim of this notebook is to take the raw model with pre-defined topic labels and assign a human-interpretable topic labels

## Load libraries

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pipeline.src.python.config as cfg
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
pd.set_option('display.max_colwidth', None)

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load model

In [3]:
MAGAZINE_1 = 'scopus'
DATASET_TEXT_FEATURE = (
    "text"  # In the dataset file, the column name that contains the text data
)
cfg_dict_1 = cfg.MAGAZINE_CONFIG[MAGAZINE_1]

In [4]:
from bertopic import BERTopic

model_path = cfg.MODELS_FOLDER / f'{MAGAZINE_1}/model_0.363_new4.safetensors'
model_1 = BERTopic.load(model_path)

2026-03-02 10:27:56,096 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


## Topic assignment

It is possible to decide which model use

### QWEN 8B

Load model

In [5]:
model_name ="Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.39it/s]


Assign label to topics

In [11]:
def evaluate_topic_assignment(keywords_list):
    messages = [
        {
            "role": "user",
            "content": (
                "Create a short topic label connecting the keywords below.\n"
                "Return ONLY the label as a short noun phrase (3–6 words).\n"
                
                "Keywords:\n"
                f"{'\n'.join(f'- {key}' for key in keywords_list)}\n\n"
            )
        }
    ]

    print(messages)


    testo = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([testo],
                              return_tensors="pt",
                              truncation=True,
                              max_length=2048).to(model.device)
    with torch.inference_mode():
        generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=24,
        do_sample = False,
        temperature = 0.0,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(generated_ids[0][len(model_inputs.input_ids[0]):],
                            skip_special_tokens=True)

    return text.strip().splitlines()[0].lstrip("-• ").strip()


### Open AI 

Define function to assign topics

In [7]:
import json
import time
from openai import OpenAI

client = OpenAI()

BULK_SCHEMA = {
    "type": "json_schema",
    "name": "topic_labels",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "label": {"type": "string",
                    "description": "Short noun phrase topic label (3–6 words)."
            }
        },
        "required": ["label"],
        "additionalProperties": False
        }
}

#"Do not add information not implied by the keywords.\n"

def evaluate_topic_assignment(
    keywords_list: list,
    model: str = "gpt-5-mini",
    max_retries: int = 3,
    sleep_between_retries: float = 0.8,
    ):

        messages = [{
            "role" : "system",
            "content" : ( "You are a semantic topic labeler.\n"
                            "Your task is to infer the underlying concept shared by a list of keywords and express it as a concise, informative noun-phrase label.\n"
                            "You must abstract from individual terms to the latent topic they collectively represent.\n"
                            "Do not explain your reasoning.\n"
                            "Return only the final label, following the specified format.\n"
                            "The noun-phrase label must have from 3 to 6 words"
                        )
                    },
                    {
            "role": "user",
            "content": (
                "Create ONE short noun-phrase label that best connects the keywords."
                f"{'\n'.join(f'- {key}' for key in keywords_list)}"
            )
        }]
        
        attempt = 0
        request_ok = False

        while (request_ok == False) & (attempt < max_retries):
            try:
                resp = client.responses.parse(
                    model=model,
                    input=messages,
                    text={"format": BULK_SCHEMA},
                )

                if resp.output_text != None:
                    data = json.loads(resp.output_text)
                    result = data["label"]
                    request_ok = True       
                
            except Exception as e:
                attempt += 1
                print(f'Request failed for the following reason:{e}')
                time.sleep(sleep_between_retries)

        return result

Assign topics

In [8]:
topic_names = {}
for i in range(0,len(set(model_1.topics_))-1):
    topic_keywords = [ keywords_pair[0] for keywords_pair in model_1.get_topic(i) ]
    topic_name = evaluate_topic_assignment(topic_keywords)
    print(f'{i}){topic_name}')
    topic_names[i] = topic_name
    

0)Multidrug-Resistant Tuberculosis Management
1)Zoonotic Spillover and Wildlife Health
2)Dengue Virus Epidemiology and Control
3)Carbapenemase Mediated Enterobacteriaceae Resistance
4)Salmonella species, serovars, strains
5)Antimicrobial Stewardship and Surveillance
6)Highly Pathogenic Avian Influenza Viruses
7)Escherichia coli recombinant protein production
8)SARS-CoV-2 Wastewater Surveillance
9)Fecal Indicator and Coliform Bacteria
10)Antibiotic Resistance in Wastewater
11)Postoperative Surgical Site Infections
12)Monkeypox (Mpox) Orthopoxvirus Infection
13)Electrochemical Biosensors for Salmonella Detection
14)Canine Leishmania Parasite Infections
15)Bovine tuberculosis caused by Mycobacterium bovis
16)West Nile virus transmission
17)SARS-CoV-2 Genomic Variant Evolution
18)Animal and Human Brucellosis
19)Leptospira species and serovar infections
20)Hospital Infection Prevention and Control
21)Vaccine Safety Surveillance Systems
22)Cleaning and Disinfection Protocols
23)Clostridioide

## Save the results

In [9]:
topic_names = { key:val.strip().replace('\n','') for key,val in topic_names.items() }

In [10]:
model_1.set_topic_labels(topic_names)

In [11]:
model_1.save(
    model_path,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)

2026-03-02 10:59:05,645 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`
